In [ ]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

In [ ]:
from faster_whisper import WhisperModel
from pathlib import Path
import librosa
import json

In [ ]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *

In [ ]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [ ]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/train"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
OUTPUT = "/workspaces/dev/datasets/asr-rankformer-datasets/LibriSpeechASRcorpus/train"
HYPERPARAMETER = "./hyperparameters/sentence_error_47_4.yml"

In [ ]:
whisper_model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")

In [ ]:
token_streamer = streamers.get_token_streamer_with_vad_v2(
    hyperparameter= HYPERPARAMETER
)

In [ ]:
src = Path(SOURCE)
output = Path(OUTPUT)
output.mkdir(parents=True, exist_ok=True)

In [ ]:
from rt_whisper.composer.data import ComposerState

def transcriber(audio) -> dict:
    param = Param()
    data=[]
    start = 0
    order = 0
    temp_order = 0
    for segment in segment_audio(audio):
        param.chunk = segment
        param.language="en"
        ctx = token_streamer.process(param, get_context=True)
        result:Result = ctx.extract()
        param.update(result)
        length = len(segment)
        data.append({
            "start": start,
            "end": start + length,
            "order": order + temp_order,
            "tokens": [{
                "start": token.start,
                "end": token.end,
                "text": token.text,
                "probability": token.probability,
            } for token in ctx.segment_tokens if token.is_word]
        })
        start = start + length
        order += sum(len(s.tokens) for s in result.completed)
        temp_order = len(ctx.state_dict[ComposerState].context.completed_tokens)

    return data

In [ ]:
def true_transcriber(audio) -> dict:
    segments, _ = whisper_model.transcribe(
        audio,
        beam_size=5,
        language="en",
        word_timestamps=True,
        vad_filter=False
    )
    segments = list(segments)
    words = [{
        "start": int(w.start * 16000),
        "end": int(w.end * 16000),
        "text": w.word,
        # "probability": w.probability
    } for segment in segments for w in segment.words]

    return words

In [ ]:
audios = list(src.rglob("*.flac"))

In [ ]:
for audio_path in audios:
    relative_path = audio_path.relative_to(src)
    target_path = output / relative_path.parent
    target_path.mkdir(parents=True, exist_ok=True)
    target_path = target_path / f"{audio_path.stem}.json"

    if target_path.exists():
        continue

    audio, _ = librosa.load(audio_path, sr=SAMPLE_RATE)
    X = transcriber(audio)
    length = [(s["order"], len(s["tokens"])) for s in X]
    Y = true_transcriber(audio)

    data = {
        "path": str(audio_path),
        "X": X,
        "Y": Y,
    }

    with open(target_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"remain {len(audios) - audios.index(audio_path)} files, {audio_path.name} done")